In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
DATASET_NAME = 'refseq_id.mirna_fc'
ID_COLUMN = 'RefSeq ID'

In [6]:
TARGETSCAN_COLUMN_TO_SEQUENCE = {
    'hsa-miR-16-5p': 'TAGCAGCACGTAAATATTGGCG', 
    'hsa-miR-106b-5p': 'TAAAGTGCTGACAGTGCAGAT', 
    'hsa-miR-200a-3p': 'TAACACTGTCTGGTAACGATGT', # https://mirbase.org/hairpin/MI0000737
    'hsa-miR-200b-3p': 'TAATACTGCCTGGTAATGATGA', 
    'hsa-miR-215-5p': 'ATGACCTATGAATTGACAGAC', 
    'hsa-let-7c-5p': 'TGAGGTAGTAGGTTGTATGGTT', 
    'hsa-miR-103a-3p': 'AGCAGCATTGTACAGGGCTATGA'
}

TS_mirnas = list(TARGETSCAN_COLUMN_TO_SEQUENCE.items())
# mirna_index = 0
# mirna_index = 1
# mirna_index = 2
# mirna_index = 3
# mirna_index = 4
mirna_index = 5
# mirna_index = 6
mirna_name = TS_mirnas[mirna_index][0]
my_miRNA = TS_mirnas[mirna_index][1]

my_miRNA, mirna_name

('TGAGGTAGTAGGTTGTATGGTT', 'hsa-let-7c-5p')

In [8]:
SEQUENCE_SOURCE_PATH = f'data/processed/GRCh37.p13 hg19/UCSC/3utr.sequences.{DATASET_NAME}.pkl'

sequence_source_df = pd.read_pickle(SEQUENCE_SOURCE_PATH)

In [11]:
sequence_source_df.columns

Index(['RefSeq ID', 'Gene symbol', 'hsa-miR-16-5p', 'hsa-miR-106b-5p',
       'hsa-miR-200a-3p', 'hsa-miR-200b-3p', 'hsa-miR-215-5p', 'hsa-let-7c-5p',
       'hsa-miR-103a-3p', 'knownGene.name', 'knownGene.chrom', 'kgAlias.kgID',
       'kgAlias.alias', 'kgXref.kgID', 'kgXref.mRNA', 'kgXref.geneSymbol',
       'kgXref.refseq', 'knownToEnsembl.name', 'knownToEnsembl.value',
       'knownToRefSeq.name', 'knownToRefSeq.value', 'ID_versioned', 'Name',
       'Description', 'sequence', 'Chromosome', 'Start', 'End', 'Strand',
       'Transcript ID', 'Representative transcript?', 'ensembl_id_no_version',
       'sequence_origin', 'utr3_length'],
      dtype='object')

In [13]:
# SCAN_MODE_EXTENSION = 'explainability_scores_'
# SCAN_MODE_EXTENSION = 'gradient_scores_'
SCAN_MODE_EXTENSION = 'grad_exp_mirbench_'

EXPLAINABILITY_SCORES_PATH = f"data/scanned/GRCh37.p13 hg19/UCSC/3utr.sequences.{DATASET_NAME}.{SCAN_MODE_EXTENSION}{mirna_name}.json"

In [14]:
explainability_scores = pd.read_json(EXPLAINABILITY_SCORES_PATH)
print("Successfully loaded as DataFrame using pd.read_json()")
print(f"Shape: {explainability_scores.shape}")
print("\nFirst few rows:")
print(explainability_scores.head())

Successfully loaded as DataFrame using pd.read_json()
Shape: (7919, 1)

First few rows:
                              TGAGGTAGTAGGTTGTATGGTT
0  [NM_000017, [0.00042925690650000005, 0.0017874...
1  [NM_000019, [0.0077499612234530006, 0.01466904...
2  [NM_000021, [0.0034649325534700003, 0.00502134...
3  [NM_000027, [0.003747011069208, 0.002840556437...
4  [NM_000030, [0.006231646053493, 0.006200858857...


In [15]:
sequence_source_df.shape[0]

8288

In [16]:
explainability_scores.shape[0]

7919

In [17]:
[print(np.array(x).shape) for x in explainability_scores.iloc[0].values[0]]

()
(560,)
(560,)
(560,)
(52, 1)


[None, None, None, None, None]

In [18]:
positives = pd.read_csv('data/chimeric_reads/positives.tsv', sep='\t', dtype={'label': int, 'chr':str, 'start': int, 'end': int})


In [22]:
positives.columns

Index(['gene', 'noncodingRNA', 'noncodingRNA_name', 'noncodingRNA_fam',
       'feature', 'label', 'chr', 'start', 'end', 'strand', 'gene_cluster_ID',
       'gene_phyloP', 'gene_phastCons', 'source_file'],
      dtype='object')

In [19]:
explainability_scores

,TGAGGTAGTAGGTTGTATGGTT
0,"[NM_000017, [0.00042925690650000005, 0.0017874..."
1,"[NM_000019, [0.0077499612234530006, 0.01466904..."
2,"[NM_000021, [0.0034649325534700003, 0.00502134..."
3,"[NM_000027, [0.003747011069208, 0.002840556437..."
4,"[NM_000030, [0.006231646053493, 0.006200858857..."
...,...
7914,"[NM_213622, [0.0006562242633660001, 0.00134102..."
7915,"[NM_213646, [0.002057584235444, 0.002441167132..."
7916,"[NM_213647, [0.002317200880497, 0.002341062063..."
7917,"[NM_213649, [0.001393726910464, 0.001541267149..."


In [21]:
def transform_ncrna_dataframe(df):
    """
    Transform a dataframe with noncoding RNA data into a structured format.
    
    Parameters:
    df (pd.DataFrame): Input dataframe with one column containing arrays of [gene_id, [numbers]]
    
    Returns:
    pd.DataFrame: Transformed dataframe with columns: noncodingRNA, gene_id, attribution
    """
    # Get the column name (noncoding RNA identifier)
    ncrna_column = df.columns[0]
    
    ncrna_list = []
    gene_id_list = []
    attribution_list = []
    
    for idx, row in df.iterrows():
        data_array = row[ncrna_column]
        
        gene_id = data_array[0]
        attribution = data_array[1]
        
        ncrna_list.append(ncrna_column)
        gene_id_list.append(gene_id)
        attribution_list.append(attribution)
    
    transformed_df = pd.DataFrame({
        'noncodingRNA': ncrna_list,
        'gene_id': gene_id_list,
        'attribution': attribution_list
    })
    
    return transformed_df

explainability_scores_formated = transform_ncrna_dataframe(explainability_scores)
print(f"Shape: {explainability_scores_formated.shape}")

print(explainability_scores_formated.head(2).to_csv(index=False))


Shape: (7919, 3)
noncodingRNA,gene_id,attribution
TGAGGTAGTAGGTTGTATGGTT,NM_000017,"[0.00042925690650000005, 0.0017874838085840002, 0.00230825971812, 0.0008751138229850001, 0.0016845751088110001, 0.001603588461875, 0.0034052918199440002, 0.004254289902746, 0.0011193959508090001, 0.0022858954034740003, 0.001207331348268, 0.0034907220397140002, 0.002850183635018, 0.0019321636646050001, 0.001283990626689, 0.000862454777234, 0.00495983089786, 0.002627057372592, 0.0030016123200760003, 0.00842499313876, 0.0016292060414950002, 0.0023932185613370002, 0.004702339792856, 0.0038470532745120004, 0.002319908167313, 0.005374161681781, 0.001827952917665, 0.003551363013684, 0.002150036588621, 0.0030210289017600003, 0.002303300338098, 0.002182511165301, 0.0026019723154600003, 0.0023921150059320003, 0.0014636967971450002, 0.002780475682811, 0.0018890555293180002, 0.0029181459976820004, 0.0042002628906620005, 0.002113438473315, 0.002292280003894, 0.002781154774129, 0.0017528381431470002, 0.00657181651331

In [22]:
positives_chr1 = positives[positives.chr == '1']
positives_chr1.head(2) # ['gene','chr','start','end'] #gene is to check the seq matches (not sure if the start/end are open/closed brackets)

,gene,noncodingRNA,noncodingRNA_name,noncodingRNA_fam,feature,label,chr,start,end,strand,gene_cluster_ID,gene_phyloP,gene_phastCons,source_file
1253320,CCCCAAATGTGGGAAACTTGACTGTATAATTTGTGGCAGTGGTAGA...,CTATACAATCTACTGTCTTTC,hsa-let-7a-3p,let-7,exon,1,1,146052098,146052147,-,34633,"[0.594, 1.343, 1.567, 1.667, 1.358, 1.343, 0.3...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.999, 0.998, 1...",test.tsv
1253321,TGTACTGTGAGATTGCCCGGTACAGCAGCAGTTGTATTCTTTATTA...,CTATACAATCTACTGTCTTTC,hsa-let-7a-3p,let-7,five_prime_utr,1,1,114749943,114749992,-,9074,"[4.404, 2.259, 4.782, 2.563, 4.21, 4.698, 1.70...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...",test.tsv


In [25]:
print(positives_chr1.head().to_csv(index=False))

gene,noncodingRNA,noncodingRNA_name,noncodingRNA_fam,feature,label,chr,start,end,strand,gene_cluster_ID,gene_phyloP,gene_phastCons,source_file
CCCCAAATGTGGGAAACTTGACTGTATAATTTGTGGCAGTGGTAGACTGC,CTATACAATCTACTGTCTTTC,hsa-let-7a-3p,let-7,exon,1,1,146052098,146052147,-,34633,"[0.594, 1.343, 1.567, 1.667, 1.358, 1.343, 0.354, -0.019, 1.343, 1.343, 1.567, 1.343, 0.366, 0.516, 1.343, 1.343, 1.567, 1.343, 1.534, 1.567, 1.567, 1.358, 0.365, 1.567, 0.364, 0.592, 1.343, 1.567, 0.592, 1.358, 1.343, 0.592, 1.567, 1.667, 1.358, 1.358, 1.358, 1.343, 1.343, 1.343, 1.567, 1.343, 0.513, 1.358, 0.364, 1.358, 0.592, 0.145, 0.592, 1.667]","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.999, 0.998, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]",test.tsv
TGTACTGTGAGATTGCCCGGTACAGCAGCAGTTGTATTCTTTATTAGCTT,CTATACAATCTACTGTCTTTC,hsa-let-7a-3p,let-7,five_prime_utr